<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import folium
import pandas as pd

In [2]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [3]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [4]:
spacex_df = pd.read_csv("data/spacex_launch_geo.csv")
spacex_df.head()

Out[0]: 
   Flight Number        Date Time (UTC)  ... class        Lat       Long
0              1  2010-06-04   18:45:00  ...     0  28.562302 -80.577356
1              2  2010-12-08   15:43:00  ...     0  28.562302 -80.577356
2              3  2012-05-22    7:44:00  ...     0  28.562302 -80.577356
3              4  2012-10-08    0:35:00  ...     0  28.562302 -80.577356
4              5  2013-03-01   15:10:00  ...     0  28.562302 -80.577356

[5 rows x 13 columns]


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [5]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

Out[0]: 
    Launch Site        Lat        Long
0   CCAFS LC-40  28.562302  -80.577356
1  CCAFS SLC-40  28.563197  -80.576820
2    KSC LC-39A  28.573255  -80.646895
3   VAFB SLC-4E  34.632834 -120.610745


,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [6]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

Out[0]: <folium.folium.Map at 0x7f7a3a9a2480>


and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


Now, let's add a circle for each launch site in data frame `launch_sites`


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [8]:
# Initialize the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add a circle and a name label for each launch site
for _, site in launch_sites_df.iterrows():
    coordinate = [site["Lat"], site["Long"]]
    site_name = site["Launch Site"]

    circle = folium.Circle(
        location=coordinate,
        radius=1000,
        color="#000000",
        fill=True,
        popup=site_name
    )
    site_map.add_child(circle)

    marker = folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{site_name}</b></div>'
        )
    )
    site_map.add_child(marker)

# Adjust the view to include all launch sites
site_map.fit_bounds(launch_sites_df[["Lat", "Long"]].values.tolist())

# Display the map
site_map

Out[0]: <folium.folium.Map at 0x7f7a3a9a21e0>


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [9]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [10]:
spacex_df.tail(10)

Out[0]: 
     Launch Site        Lat       Long  class
46    KSC LC-39A  28.573255 -80.646895      1
47    KSC LC-39A  28.573255 -80.646895      1
48    KSC LC-39A  28.573255 -80.646895      1
49  CCAFS SLC-40  28.563197 -80.576820      1
50  CCAFS SLC-40  28.563197 -80.576820      1
51  CCAFS SLC-40  28.563197 -80.576820      0
52  CCAFS SLC-40  28.563197 -80.576820      0
53  CCAFS SLC-40  28.563197 -80.576820      0
54  CCAFS SLC-40  28.563197 -80.576820      1
55  CCAFS SLC-40  28.563197 -80.576820      0


,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [11]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [12]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
spacex_df["marker_color"] = spacex_df["class"].apply(
    lambda outcome: "green" if outcome == 1 else "red"
)

spacex_df.tail()

Out[0]: 
     Launch Site        Lat      Long  class marker_color
51  CCAFS SLC-40  28.563197 -80.57682      0          red
52  CCAFS SLC-40  28.563197 -80.57682      0          red
53  CCAFS SLC-40  28.563197 -80.57682      0          red
54  CCAFS SLC-40  28.563197 -80.57682      1        green
55  CCAFS SLC-40  28.563197 -80.57682      0          red


,Launch Site,Lat,Long,class,marker_color
51,CCAFS SLC-40,28.563197,-80.57682,0,red
52,CCAFS SLC-40,28.563197,-80.57682,0,red
53,CCAFS SLC-40,28.563197,-80.57682,0,red
54,CCAFS SLC-40,28.563197,-80.57682,1,green
55,CCAFS SLC-40,28.563197,-80.57682,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [13]:
# Add the cluster to the map
site_map.add_child(marker_cluster)

# Create a separate marker for every launch record
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record["Lat"], record["Long"]],
        popup=f'{record["Launch Site"]} | Class: {record["class"]}',
        icon=folium.Icon(
            color="white",
            icon_color=record["marker_color"],
            icon="info-sign"
        )
    )

    marker_cluster.add_child(marker)

# Display the map
site_map

Out[0]: <folium.folium.Map at 0x7f7a3a9a21e0>


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [14]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [15]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Out[0]: <folium.folium.Map at 0x7f7a3a9a21e0>


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [16]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [17]:
# Select CCAFS SLC-40 from the launch-site table
selected_site = launch_sites_df[
    launch_sites_df["Launch Site"] == "CCAFS SLC-40"
].iloc[0]

launch_site_lat = selected_site["Lat"]
launch_site_lon = selected_site["Long"]

# Example coastline point supplied in the lab
coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon,
    coastline_lat, coastline_lon
)

print(f"Distance to selected coastline point: {distance_coastline:.2f} km")

Distance to selected coastline point: 0.51 km


In [18]:
distance_marker = folium.Marker(
    location=[coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(160, 30),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size:12px; color:#d35400;">'
            f'<b>{distance_coastline:.2f} km</b></div>'
        )
    )
)

site_map.add_child(distance_marker)

Out[0]: <folium.folium.Map at 0x7f7a3a9a21e0>


In [19]:
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

lines = folium.PolyLine(
    locations=coordinates,
    color="blue",
    weight=2
)

site_map.add_child(lines)

# Zoom to the two points
site_map.fit_bounds(coordinates)

site_map

Out[0]: <folium.folium.Map at 0x7f7a3a9a21e0>


*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [20]:
def add_distance_marker(name, latitude, longitude):
    distance = calculate_distance(
        launch_site_lat, launch_site_lon,
        latitude, longitude
    )

    folium.Marker(
        location=[latitude, longitude],
        popup=f"{name}: {distance:.2f} km",
        icon=DivIcon(
            icon_size=(200, 30),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:12px; color:#d35400;">'
                f'<b>{name}: {distance:.2f} km</b></div>'
            )
        )
    ).add_to(site_map)

    folium.PolyLine(
        locations=[
            [launch_site_lat, launch_site_lon],
            [latitude, longitude]
        ],
        color="blue",
        weight=2
    ).add_to(site_map)

    return distance

In [21]:
# Selected mapped points for comparison, not minimum distances to these features.
# OSM nodes 1116487179 (Centaur Road) and 1116487198 (rail spur).
# GeoNames 4175437 identifies the Titusville city reference point.
points = {
    "Centaur Road": (28.5615810, -80.5772710),
    "Rail spur": (28.5618912, -80.5774060),
    "Titusville": (28.61222, -80.80755),
}
distances = {name: add_distance_marker(name, *coords) for name, coords in points.items()}
distances["Selected coastline point"] = distance_coastline
pd.Series(distances, name="Great-circle km").round(2)

Out[0]: 
Centaur Road                 0.19
Rail spur                    0.16
Titusville                  23.19
Selected coastline point     0.51
Name: Great-circle km, dtype: float64


Centaur Road                 0.19
Rail spur                    0.16
Titusville                  23.19
Selected coastline point     0.51
Name: Great-circle km, dtype: float64

In [22]:
site_map.fit_bounds([[launch_site_lat, launch_site_lon], *points.values(),
                     [coastline_lat, coastline_lon]])
site_map

Out[0]: <folium.folium.Map at 0x7f7a3a9a21e0>


The selected road and rail points lie close to CCAFS SLC-40. Titusville’s city reference point is much farther away. These are distances to specific points, not minimum distances to a feature or to a city boundary. Map data © OpenStreetMap contributors.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
